# Gold Layer - Customer Lifecycle Metrics View

## Purpose
Provide real-time overall customer lifecycle KPIs for executive dashboards without materializing data.

## Type
**SQL View** (not a materialized table)

## Output
* **View:** `big_data.gold.vw_customer_lifecycle_metrics`
* **Rows:** 1 (single aggregated row)

## Usage
```sql
SELECT * FROM big_data.gold.vw_customer_lifecycle_metrics;
```

In [0]:
%sql
-- Create or replace view: vw_customer_lifecycle_metrics

CREATE OR REPLACE VIEW big_data.gold.vw_customer_lifecycle_metrics AS
WITH customer_metrics AS (
  SELECT 
    o.user_id,
    COUNT(DISTINCT o.order_id) AS total_orders,
    ROUND(SUM(p.price_usd), 2) AS lifetime_value_usd
  FROM big_data.silver.orders o
  JOIN big_data.silver.order_products op ON o.order_id = op.order_id
  JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
  GROUP BY o.user_id
)
SELECT 
  COUNT(DISTINCT user_id) AS total_customers,
  ROUND(AVG(total_orders), 2) AS avg_orders_per_customer,
  ROUND(AVG(lifetime_value_usd), 2) AS avg_customer_lifetime_value_usd
FROM customer_metrics;

In [0]:
%sql
-- Verify view exists and preview KPIs
SELECT * FROM big_data.gold.vw_customer_lifecycle_metrics;